# Emotive TTS — Colab Pipeline

**COMP3065 Project** — Emotion-Conditioned Neural TTS via VITS Fine-Tuning

| Section | Task | Systems |
|---------|------|---------|
| 0 | Setup — Mount Drive, clone repo, install deps | — |
| 1 | Data preparation — Download & process EmoV-DB | — |
| 2 | Train System A — Domain adaptation (no emotion) | A |
| 3 | Train System B — + emotion embedding | B |
| 4 | Train System C — + prosody heads | C |
| 5 | Inference — Generate eval stimuli (A0/A/B/C) | All |
| 6 | Evaluation — Prosody + SER + plots | All |

**Runtime:** GPU → T4 (Runtime → Change runtime type → T4 GPU)  
**GitHub:** https://github.com/jimmy00415/COMP3067_Project

---

> **Resumability:** Every stage saves checkpoints to Google Drive.  
> If the session disconnects, **Runtime → Run all** — completed stages are skipped automatically.

## 0. Setup

In [ ]:
# ══ 0a. Mount Google Drive ══
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_PROJECT = '/content/drive/MyDrive/COMP3065_Project'
for subdir in ['checkpoints/system_a', 'checkpoints/system_b',
               'checkpoints/system_c', 'data/processed', 'data/raw',
               'data/manifests', 'tables', 'figures', 'outputs']:
    os.makedirs(os.path.join(DRIVE_PROJECT, subdir), exist_ok=True)

print(f'Drive project folder: {DRIVE_PROJECT}')

In [ ]:
# ══ 0b. Clone / pull project from GitHub ══
import os

REPO_URL = 'https://github.com/jimmy00415/COMP3067_Project.git'
LOCAL_PROJECT = '/content/COMP3067_Project'

if os.path.exists(os.path.join(LOCAL_PROJECT, '.git')):
    print('Repo exists — pulling latest...')
    !cd {LOCAL_PROJECT} && git pull --ff-only
else:
    if os.path.exists(LOCAL_PROJECT):
        import shutil
        shutil.rmtree(LOCAL_PROJECT)
    !git clone {REPO_URL} {LOCAL_PROJECT}

%cd {LOCAL_PROJECT}
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ══ 0c. Install dependencies ══
#
# Strategy:
#   1. Check if TTS is already importable (post-restart -> skip install)
#   2. If not: install espeak-ng, coqui-tts, pin numpy<2, restart runtime
#   3. After restart: Runtime -> Run all continues from here and skips
#
import sys, os

def _deps_ok():
    """Return True if all critical packages load without C-ABI crash."""
    try:
        import numpy
        numpy.array([1.0]).astype(numpy.float32)   # C-ABI smoke test
        import TTS
        import librosa
        import soundfile
        import pandas
        import scipy
        import sklearn
        return True
    except Exception:
        return False

if _deps_ok():
    import TTS, numpy
    print(f'All deps OK — TTS {TTS.__version__}, numpy {numpy.__version__}')
else:
    print(f'Python {sys.version}')

    # 1. System dep: espeak-ng (survives restarts)
    print('[1/4] Installing espeak-ng ...')
    !apt-get install -y -qq espeak-ng > /dev/null 2>&1

    # 2. Pin numpy<2 BEFORE coqui-tts (prevents 2.x from being pulled)
    print('[2/4] Installing numpy<2 + coqui-tts ...')
    !pip install -q 'numpy>=1.24,<2.0'
    !pip install -q coqui-tts 'numpy>=1.24,<2.0'
    # Re-pin numpy in case coqui-tts pulled 2.x transitively
    !pip install -q 'numpy>=1.24,<2.0'

    # 3. Remaining deps
    print('[3/4] Installing remaining deps ...')
    !pip install -q 'speechbrain>=1.0' 'omegaconf>=2.3' 'mlflow>=2.10'
    !pip install -q 'pandas>=2.0' 'matplotlib>=3.7' 'seaborn>=0.13'
    !pip install -q 'scipy>=1.11' 'scikit-learn>=1.3' 'soundfile>=0.12'
    !pip install -q 'librosa>=0.10' 'gradio>=4.0'

    # 4. Install project in editable mode
    !pip install -q -e .

    # Final numpy pin (belt-and-suspenders)
    !pip install -q 'numpy>=1.24,<2.0'

    print()
    print('[4/4] Restarting runtime to load numpy C extensions cleanly ...')
    print('      After restart, click  Runtime -> Run all  to continue.')
    os.kill(os.getpid(), 9)  # hard restart

In [ ]:
# ══ 0d. Post-restart re-init ══
#
# After runtime restart ALL Python state is lost.
# This cell re-establishes every variable the rest of the notebook needs.
#
import os, sys, shutil, logging, gc

# ---------- constants (must match 0a / 0b) ----------
DRIVE_PROJECT  = '/content/drive/MyDrive/COMP3065_Project'
LOCAL_PROJECT  = '/content/COMP3067_Project'

# Mount Drive if not already mounted (happens after restart)
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# Make sure we are in the repo directory
os.chdir(LOCAL_PROJECT)
if LOCAL_PROJECT not in sys.path:
    sys.path.insert(0, LOCAL_PROJECT)

# Install project package if needed (editable mode survives restart)
try:
    from src.data.utils import EMOTION_MAP, EMOTION_LABELS
except ImportError:
    os.system(f'{sys.executable} -m pip install -q -e .')
    from src.data.utils import EMOTION_MAP, EMOTION_LABELS

# Logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s',
                    force=True)

# Verify
import torch, numpy as np
print(f'PyTorch  : {torch.__version__}')
print(f'numpy    : {np.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU      : {gpu}  ({vram:.1f} GB)')
else:
    print('WARNING: No GPU detected — training will be very slow.')
print(f'Emotions : {EMOTION_LABELS}')
print(f'CWD      : {os.getcwd()}')

## 1. Data Preparation

In [ ]:
# ══ 1a. Download EmoV-DB from OpenSLR ══
import os, glob, tarfile, shutil

EMOVDB_DRIVE = os.path.join(DRIVE_PROJECT, 'data', 'raw', 'EmoV-DB')
EMOVDB_LOCAL = 'data/raw/EmoV-DB'
os.makedirs('data/raw', exist_ok=True)

def _count_wavs(d):
    if not os.path.isdir(d):
        return 0
    return len(glob.glob(os.path.join(d, '**', '*.wav'), recursive=True))

MIN_WAVS = 100  # EmoV-DB has ~6000+ wavs

# Priority 1: already local with enough files
if os.path.isdir(EMOVDB_LOCAL) and _count_wavs(EMOVDB_LOCAL) >= MIN_WAVS:
    print(f'EmoV-DB found locally ({_count_wavs(EMOVDB_LOCAL)} wavs)')

# Priority 2: Drive cache
elif os.path.isdir(EMOVDB_DRIVE) and _count_wavs(EMOVDB_DRIVE) >= MIN_WAVS:
    if not os.path.exists(EMOVDB_LOCAL):
        os.symlink(EMOVDB_DRIVE, EMOVDB_LOCAL)
    print(f'EmoV-DB symlinked from Drive ({_count_wavs(EMOVDB_LOCAL)} wavs)')

# Priority 3: download
else:
    print('Downloading EmoV-DB from OpenSLR (may take 10-15 min) ...')
    os.makedirs(EMOVDB_LOCAL, exist_ok=True)

    BASE_URL = 'https://www.openslr.org/resources/115'
    speakers  = ['bea', 'jenie', 'josh', 'sam']
    emotions  = {'Amused': 'Amused', 'Angry': 'Angry',
                 'Disgusted': 'Disgust', 'Neutral': 'Neutral'}

    for emo_slr, emo_folder in emotions.items():
        emo_dir = os.path.join(EMOVDB_LOCAL, emo_folder)
        os.makedirs(emo_dir, exist_ok=True)
        for spk in speakers:
            fname    = f'{spk}_{emo_slr}.tar.gz'
            tar_path = f'/tmp/{fname}'
            if not os.path.isfile(tar_path) or os.path.getsize(tar_path) < 10_000:
                ret = os.system(f'wget -q --show-progress -O {tar_path} {BASE_URL}/{fname}')
                if ret != 0 or not os.path.isfile(tar_path) or os.path.getsize(tar_path) < 10_000:
                    print(f'  skip {fname} (not available)')
                    if os.path.isfile(tar_path):
                        os.remove(tar_path)
                    continue
            try:
                with tarfile.open(tar_path, 'r:gz') as tf:
                    tf.extractall(emo_dir)
                print(f'  extracted {fname} -> {emo_folder}/')
            except Exception as e:
                print(f'  WARN: {fname}: {e}')

    # Persist to Drive for next session
    print('Persisting to Drive ...')
    if not os.path.isdir(EMOVDB_DRIVE):
        os.makedirs(os.path.dirname(EMOVDB_DRIVE), exist_ok=True)
        shutil.copytree(EMOVDB_LOCAL, EMOVDB_DRIVE)
    print(f'EmoV-DB ready ({_count_wavs(EMOVDB_LOCAL)} wavs)')

# Quick audit
for d in sorted(os.listdir(EMOVDB_LOCAL)):
    p = os.path.join(EMOVDB_LOCAL, d)
    if os.path.isdir(p):
        print(f'  {d}: {_count_wavs(p)} wav files')

In [ ]:
# ══ 1b. Run data preparation pipeline ══
import os, yaml, shutil, glob
from pathlib import Path

MANIFESTS_DIR = 'data/manifests'
PROCESSED_DIR = 'data/processed/train'
train_csv = os.path.join(MANIFESTS_DIR, 'train.csv')

# Check Drive for existing manifests first
drive_train_csv = os.path.join(DRIVE_PROJECT, 'data', 'manifests', 'train.csv')
if not os.path.isfile(train_csv) and os.path.isfile(drive_train_csv):
    os.makedirs(MANIFESTS_DIR, exist_ok=True)
    drive_manifests = os.path.join(DRIVE_PROJECT, 'data', 'manifests')
    for f in Path(drive_manifests).glob('*.csv'):
        shutil.copy2(str(f), MANIFESTS_DIR)
    print('Restored manifests from Drive.')

# Also restore processed audio from Drive if available locally
drive_processed = os.path.join(DRIVE_PROJECT, 'data', 'processed', 'train')
if not os.path.isdir(PROCESSED_DIR) or len(glob.glob(os.path.join(PROCESSED_DIR, '**/*.wav'), recursive=True)) < 10:
    if os.path.isdir(drive_processed) and len(glob.glob(os.path.join(drive_processed, '**/*.wav'), recursive=True)) > 100:
        print('Restoring processed audio from Drive ...')
        os.makedirs(PROCESSED_DIR, exist_ok=True)
        shutil.copytree(drive_processed, PROCESSED_DIR, dirs_exist_ok=True)
        n_restored = len(glob.glob(os.path.join(PROCESSED_DIR, '**/*.wav'), recursive=True))
        print(f'Restored {n_restored} processed audio files from Drive.')

# Validate that processed audio actually exists (not just the CSV)
_audio_ok = False
if os.path.isfile(train_csv):
    import pandas as pd
    df = pd.read_csv(train_csv)
    _pcol = 'processed_path' if 'processed_path' in df.columns else 'file_path'
    if _pcol in df.columns:
        _n_exist = df[_pcol].head(20).apply(lambda p: os.path.isfile(str(p))).sum()
        _audio_ok = _n_exist > 0
    if _audio_ok:
        print(f'Manifests + audio OK ({len(df)} samples) \u2014 skipping data prep.')
        print(df['emotion'].value_counts().to_string())
    else:
        print('Manifest exists but processed audio files are missing \u2014 re-running data prep.')

if not _audio_ok:
    from src.data.prepare import prepare_dataset

    with open('configs/data.yaml', 'r') as f:
        data_cfg = yaml.safe_load(f)

    data_cfg['dataset']['raw_dir'] = EMOVDB_LOCAL

    summary = prepare_dataset(data_cfg)
    print('Dataset summary:')
    for k, v in summary.items():
        print(f'  {k}: {v}')

    # Persist manifests to Drive
    drive_manifests = os.path.join(DRIVE_PROJECT, 'data', 'manifests')
    os.makedirs(drive_manifests, exist_ok=True)
    for f in Path(MANIFESTS_DIR).glob('*.csv'):
        shutil.copy2(str(f), drive_manifests)
    print('Manifests saved to Drive.')

    # Persist processed audio to Drive for future sessions
    if os.path.isdir(PROCESSED_DIR):
        if not os.path.isdir(drive_processed) or len(glob.glob(os.path.join(drive_processed, '**/*.wav'), recursive=True)) < 100:
            print('Saving processed audio to Drive (first time only) ...')
            os.makedirs(drive_processed, exist_ok=True)
            shutil.copytree(PROCESSED_DIR, drive_processed, dirs_exist_ok=True)
            n_saved = len(glob.glob(os.path.join(drive_processed, '**/*.wav'), recursive=True))
            print(f'Saved {n_saved} processed audio files to Drive.')


## 2. Training: System A — Domain Adaptation

Fine-tunes pretrained VITS on EmoV-DB **without** emotion labels.
This is the domain-adapted baseline that isolates the conditioning effect (A → B).

In [ ]:
# ══ 2a. Train System A ══
import os, yaml, torch, shutil, gc

CKPT_A_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints', 'system_a', 'best.pth')
CKPT_A_LOCAL = 'checkpoints/system_a/best.pth'
os.makedirs(os.path.dirname(CKPT_A_LOCAL), exist_ok=True)

if os.path.isfile(CKPT_A_DRIVE) and not os.path.isfile(CKPT_A_LOCAL):
    shutil.copy2(CKPT_A_DRIVE, CKPT_A_LOCAL)
    print(f'System A checkpoint restored from Drive.')

if os.path.isfile(CKPT_A_LOCAL):
    sz = os.path.getsize(CKPT_A_LOCAL) / 1e6
    print(f'System A checkpoint exists ({sz:.1f} MB) — SKIPPING training.')
else:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from src.training.train import train

    with open('configs/train_a.yaml', 'r') as f:
        cfg_a = yaml.safe_load(f)

    # Colab-safe overrides
    cfg_a['use_cuda'] = torch.cuda.is_available()
    cfg_a.setdefault('training', {})
    cfg_a['training']['fp16'] = torch.cuda.is_available()
    cfg_a['training']['num_workers'] = 0
    cfg_a['training']['init_from'] = 'pretrained'
    cfg_a['training'].setdefault('checkpoint', {})['save_dir'] = os.path.dirname(CKPT_A_LOCAL)

    # Reduce batch size if VRAM < 14 GB
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if vram_gb < 14:
            cfg_a['training']['batch_size'] = 8
            print(f'Low VRAM ({vram_gb:.1f} GB) — reducing batch_size to 8')

    # Auto-backup best checkpoints to Drive during training
    cfg_a['training']['drive_checkpoint_dir'] = os.path.dirname(CKPT_A_DRIVE)

    print('Starting System A training ...')
    result_a = train(cfg_a)
    print(f'System A training complete: {result_a}')

    # Final Drive backup
    if os.path.isfile(CKPT_A_LOCAL):
        os.makedirs(os.path.dirname(CKPT_A_DRIVE), exist_ok=True)
        shutil.copy2(CKPT_A_LOCAL, CKPT_A_DRIVE)
        print('System A checkpoint saved to Drive.')

    del result_a
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 3. Training: System B — Emotion Embedding

System B = System A + `nn.Embedding(4, 192)` for emotion conditioning.
Warm-starts from the System A checkpoint.

In [ ]:
# ══ 3a. Train System B ══
import os, yaml, torch, shutil, gc

CKPT_A_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints', 'system_a', 'best.pth')
CKPT_A_LOCAL = 'checkpoints/system_a/best.pth'
CKPT_B_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints', 'system_b', 'best.pth')
CKPT_B_LOCAL = 'checkpoints/system_b/best.pth'
os.makedirs(os.path.dirname(CKPT_B_LOCAL), exist_ok=True)

if os.path.isfile(CKPT_B_DRIVE) and not os.path.isfile(CKPT_B_LOCAL):
    shutil.copy2(CKPT_B_DRIVE, CKPT_B_LOCAL)
    print('System B checkpoint restored from Drive.')

if os.path.isfile(CKPT_B_LOCAL):
    sz = os.path.getsize(CKPT_B_LOCAL) / 1e6
    print(f'System B checkpoint exists ({sz:.1f} MB) — SKIPPING training.')
else:
    if not os.path.isfile(CKPT_A_LOCAL) and os.path.isfile(CKPT_A_DRIVE):
        shutil.copy2(CKPT_A_DRIVE, CKPT_A_LOCAL)
    if not os.path.isfile(CKPT_A_LOCAL):
        raise FileNotFoundError(
            f'System A checkpoint not found at {CKPT_A_LOCAL}. '
            'Train System A first (Section 2).'
        )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from src.training.train import train

    with open('configs/train_b.yaml', 'r') as f:
        cfg_b = yaml.safe_load(f)

    cfg_b['use_cuda'] = torch.cuda.is_available()
    cfg_b.setdefault('training', {})
    cfg_b['training']['fp16'] = torch.cuda.is_available()
    cfg_b['training']['num_workers'] = 0
    cfg_b['training']['init_from'] = CKPT_A_LOCAL
    cfg_b['training'].setdefault('checkpoint', {})['save_dir'] = os.path.dirname(CKPT_B_LOCAL)

    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if vram_gb < 14:
            cfg_b['training']['batch_size'] = 8
            print(f'Low VRAM ({vram_gb:.1f} GB) — reducing batch_size to 8')

    cfg_b['training']['drive_checkpoint_dir'] = os.path.dirname(CKPT_B_DRIVE)

    print(f'Warm-starting from System A: {CKPT_A_LOCAL}')
    print('Starting System B training ...')
    result_b = train(cfg_b)
    print(f'System B training complete: {result_b}')

    if os.path.isfile(CKPT_B_LOCAL):
        os.makedirs(os.path.dirname(CKPT_B_DRIVE), exist_ok=True)
        shutil.copy2(CKPT_B_LOCAL, CKPT_B_DRIVE)
        print('System B checkpoint saved to Drive.')

    del result_b
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 4. Training: System C — Prosody Supervision

System C = System B + utterance-level prosody auxiliary heads (F0 + energy).
Warm-starts from the System B checkpoint. Loss weight λ = 0.1.

In [ ]:
# ══ 4a. Train System C ══
import os, yaml, torch, shutil, gc

CKPT_B_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints', 'system_b', 'best.pth')
CKPT_B_LOCAL = 'checkpoints/system_b/best.pth'
CKPT_C_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints', 'system_c', 'best.pth')
CKPT_C_LOCAL = 'checkpoints/system_c/best.pth'
os.makedirs(os.path.dirname(CKPT_C_LOCAL), exist_ok=True)

if os.path.isfile(CKPT_C_DRIVE) and not os.path.isfile(CKPT_C_LOCAL):
    shutil.copy2(CKPT_C_DRIVE, CKPT_C_LOCAL)
    print('System C checkpoint restored from Drive.')

if os.path.isfile(CKPT_C_LOCAL):
    sz = os.path.getsize(CKPT_C_LOCAL) / 1e6
    print(f'System C checkpoint exists ({sz:.1f} MB) — SKIPPING training.')
else:
    if not os.path.isfile(CKPT_B_LOCAL) and os.path.isfile(CKPT_B_DRIVE):
        shutil.copy2(CKPT_B_DRIVE, CKPT_B_LOCAL)
    if not os.path.isfile(CKPT_B_LOCAL):
        raise FileNotFoundError(
            f'System B checkpoint not found at {CKPT_B_LOCAL}. '
            'Train System B first (Section 3).'
        )

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from src.training.train import train

    with open('configs/train_c.yaml', 'r') as f:
        cfg_c = yaml.safe_load(f)

    cfg_c['use_cuda'] = torch.cuda.is_available()
    cfg_c.setdefault('training', {})
    cfg_c['training']['fp16'] = torch.cuda.is_available()
    cfg_c['training']['num_workers'] = 0
    cfg_c['training']['init_from'] = CKPT_B_LOCAL
    cfg_c['training'].setdefault('checkpoint', {})['save_dir'] = os.path.dirname(CKPT_C_LOCAL)

    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        if vram_gb < 14:
            cfg_c['training']['batch_size'] = 8
            print(f'Low VRAM ({vram_gb:.1f} GB) — reducing batch_size to 8')

    cfg_c['training']['drive_checkpoint_dir'] = os.path.dirname(CKPT_C_DRIVE)

    print(f'Warm-starting from System B: {CKPT_B_LOCAL}')
    print('Starting System C training ...')
    result_c = train(cfg_c)
    print(f'System C training complete: {result_c}')

    if os.path.isfile(CKPT_C_LOCAL):
        os.makedirs(os.path.dirname(CKPT_C_DRIVE), exist_ok=True)
        shutil.copy2(CKPT_C_LOCAL, CKPT_C_DRIVE)
        print('System C checkpoint saved to Drive.')

    del result_c
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'GPU memory freed: {torch.cuda.memory_allocated()/1e9:.2f} GB in use')

## 5. Inference

Generate evaluation stimuli: **16 canary texts × 4 emotions × 4 systems = 256 samples**

In [ ]:
# ══ 5a. Generate evaluation stimuli for all 4 systems ══
import os, yaml, torch, shutil, gc

EVAL_MANIFEST = 'data/processed/eval_stimuli/eval_manifest.csv'

drive_manifest = os.path.join(DRIVE_PROJECT, 'data', 'processed', 'eval_stimuli', 'eval_manifest.csv')
if not os.path.isfile(EVAL_MANIFEST) and os.path.isfile(drive_manifest):
    drive_eval = os.path.join(DRIVE_PROJECT, 'data', 'processed', 'eval_stimuli')
    os.makedirs('data/processed/eval_stimuli', exist_ok=True)
    shutil.copytree(drive_eval, 'data/processed/eval_stimuli', dirs_exist_ok=True)
    print('Restored eval stimuli from Drive.')

if os.path.isfile(EVAL_MANIFEST):
    import pandas as pd
    n = len(pd.read_csv(EVAL_MANIFEST))
    print(f'Eval manifest exists ({n} entries) — SKIPPING inference.')
else:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    from src.inference.run import run_inference

    with open('configs/infer.yaml', 'r') as f:
        infer_cfg = yaml.safe_load(f)

    infer_cfg.setdefault('inference', {})
    infer_cfg['inference']['use_cuda'] = torch.cuda.is_available()
    infer_cfg['inference']['systems'] = ['A0', 'A', 'B', 'C']
    infer_cfg['inference']['output_dir'] = 'data/processed/eval_stimuli'
    infer_cfg['inference']['canary_texts'] = 'configs/canary_texts.txt'
    infer_cfg['inference']['system_a_checkpoint'] = 'checkpoints/system_a/best.pth'
    infer_cfg['inference']['system_a_config']     = 'checkpoints/system_a/config.json'
    infer_cfg['inference']['system_b_checkpoint'] = 'checkpoints/system_b/best.pth'
    infer_cfg['inference']['system_c_checkpoint'] = 'checkpoints/system_c/best.pth'

    print('Generating evaluation samples for all 4 systems ...')
    results = run_inference(infer_cfg)
    print(f'Inference complete: {results.get("total_files", 0)} files')

    eval_drive = os.path.join(DRIVE_PROJECT, 'data', 'processed', 'eval_stimuli')
    os.makedirs(eval_drive, exist_ok=True)
    shutil.copytree('data/processed/eval_stimuli', eval_drive, dirs_exist_ok=True)
    print('Eval stimuli saved to Drive.')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 6. Evaluation

- **6a** Prosody analysis (PRIMARY metric — F0 & energy stats)
- **6b** SER probe (AUXILIARY metric — label-mismatch caveat)
- **6c** Plots (publication-quality figures)
- **6d** Listening test stimulus pack
- **6e** Save all outputs to Drive

In [ ]:
# ══ 6a. Prosody analysis (PRIMARY metric) ══
import os, yaml

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

print('Running prosody evaluation ...')
try:
    from src.evaluation.prosody import run_prosody_evaluation
    prosody_results = run_prosody_evaluation(eval_cfg)
    print('Prosody evaluation complete.')
    if 'agg_stats' in prosody_results:
        from IPython.display import display
        display(prosody_results['agg_stats'])
except Exception as e:
    print(f'Prosody analysis failed: {e}')

In [ ]:
# ══ 6b. SER probe (AUXILIARY metric — non-critical) ══
import yaml

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

print('Running SER probe evaluation ...')
try:
    from src.evaluation.ser_probe import run_ser_evaluation
    ser_results = run_ser_evaluation(eval_cfg)
    agreement = ser_results.get('ser_proxy_agreement', 0)
    n_excluded = ser_results.get('n_excluded', 0)
    print(f'SER proxy agreement: {agreement:.2%}')
    print(f'  NOTE: This is an AUXILIARY metric only (Wav2Vec2-based proxy).')
    if n_excluded > 0:
        print(f'  Excluded {n_excluded} samples with unmapped emotions (disgust).')
except Exception as e:
    print(f'SER probe failed (non-critical): {e}')
    print('Continuing without SER results — prosody is the primary metric.')

In [ ]:
# ══ 6c. Generate all evaluation plots ══
import os, yaml

with open('configs/eval.yaml', 'r') as f:
    eval_cfg = yaml.safe_load(f)

print('Generating plots ...')
try:
    from src.evaluation.plots import generate_all_plots
    generate_all_plots(eval_cfg)
    print('All plots saved to figures/')
except Exception as e:
    print(f'Plot generation failed: {e}')

# Display key plots inline
from IPython.display import Image, display as ipy_display
for plot_name in ['f0_by_system_emotion.png', 'prosody_comparison_grid.png']:
    plot_path = os.path.join('figures', plot_name)
    if os.path.isfile(plot_path):
        print(f'\n--- {plot_name} ---')
        ipy_display(Image(filename=plot_path, width=800))
    else:
        print(f'  ({plot_name} not generated)')

In [ ]:
# ══ 6d. Listening test stimulus pack ══
import os

EVAL_MANIFEST = 'data/processed/eval_stimuli/eval_manifest.csv'
if os.path.isfile(EVAL_MANIFEST):
    try:
        from src.evaluation.listening_test import create_stimulus_pack
        print('Creating listening test stimulus pack ...')
        stim_summary = create_stimulus_pack(
            manifest_path=EVAL_MANIFEST,
            output_dir='outputs/listening_test',
        )
        print(f'Stimulus pack: {stim_summary}')
    except Exception as e:
        print(f'Listening test pack failed (non-critical): {e}')
else:
    print('No eval manifest — run Section 5 first.')

In [ ]:
# ══ 6e. Save ALL outputs to Google Drive ══
import os, shutil

for folder in ['tables', 'figures', 'outputs', 'docs']:
    if os.path.isdir(folder):
        dst = os.path.join(DRIVE_PROJECT, folder)
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        print(f'  {folder}/ -> Drive')
    else:
        print(f'  {folder}/ not found, skipping')

print('\nAll outputs saved to Google Drive.')

## 7. Summary

| Step | Section | Description |
|------|---------|-------------|
| Data prep | S1 | EmoV-DB downloaded, processed, splits created |
| System A | S2 | Domain-adapted VITS (no emotion labels) |
| System B | S3 | + emotion embedding (nn.Embedding(4, 192)) |
| System C | S4 | + utterance-level prosody heads (F0 + energy, λ=0.1) |
| Inference | S5 | 16 texts × 4 emotions × 4 systems = 256 samples |
| Prosody | S6a | PRIMARY — F0/energy stats + statistical tests |
| SER probe | S6b | AUXILIARY — Wav2Vec2 proxy agreement |
| Plots | S6c | Publication-quality figures |
| Listening test | S6d | Stimulus pack for human evaluation |

**Checkpoints and outputs** are persisted to Google Drive under
`/content/drive/MyDrive/COMP3065_Project/`

> **Resumability:** Re-run from top after disconnect — completed stages
> are automatically detected and skipped via checkpoint presence checks.